In [ ]:
import json
import shutil
from pathlib import Path
from collections import Counter

## Re-labelling function

`relabel_json_dir()` reads every LabelMe JSON in an input folder, swaps the labels
according to a `label_map` dictionary (old label → new label), and writes the
modified JSONs to an output folder with the same filenames. The originals are
never touched.

- Any label **not** in `label_map` is left unchanged (and a warning is printed so nothing slips through silently).
- A summary of how many labels were converted (and from what to what) is printed at the end.
- Optionally, the matching PNG for each JSON can be copied across too (LabelMe expects the image next to the JSON).
- `on_exists` controls what happens when a file with the same name is already in the output folder:
  - `"overwrite"` (default) — replace it; safe when re-running the same conversion
  - `"skip"` — leave existing files alone and list them in the summary
  - `"error"` — check for clashes up front and stop before writing anything; safest when different source folders share filenames (e.g. `WH` and `waterhole_river` both contain `2024-04_example_S2.png`)

In [ ]:
def relabel_json_dir(input_dir, output_dir, label_map, copy_images=False,
                     on_exists="skip"):
    """
    Re-label LabelMe JSON annotation files and save them to a new folder.

    Parameters
    ----------
    input_dir : str or Path
        Folder containing the original LabelMe JSON files.
    output_dir : str or Path
        Folder to save the re-labelled JSON files to (created if it
        doesn't exist). Filenames are kept the same, so the originals
        in `input_dir` are never overwritten.
    label_map : dict
        Mapping of old label -> new label, e.g.
            {"WH": "waterhole"}
        or a many-to-one mapping, e.g.
            {"WH_wet": "waterhole", "Dry_WH": "waterhole", ...}
        Labels not present in the map are left unchanged (a warning
        is printed so you can spot unexpected labels).
    copy_images : bool, optional
        If True, also copy the image referenced by each JSON's
        "imagePath" into `output_dir` (LabelMe needs the image
        alongside the JSON to display it). Default False.
    on_exists : str, optional
        What to do when a file with the same name already exists in
        `output_dir`:
          - "overwrite" (default): replace it (safe when re-running
            the same conversion; identical files are just regenerated)
          - "skip": leave the existing file alone and report it
          - "error": stop BEFORE writing anything and list the
            clashing filenames (safest when different source folders
            share filenames, so nothing is clobbered unnoticed)
    """
    if on_exists not in ("overwrite", "skip", "error"):
        raise ValueError(f"on_exists must be 'overwrite', 'skip' or 'error', got '{on_exists}'")

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)  # create output folder if needed

    json_paths = sorted(input_dir.glob("*.json"))

    # For "error" mode, check ALL potential clashes up front and stop before
    # writing anything, so a collision can never leave a half-written folder
    if on_exists == "error":
        clashes = []
        for json_path in json_paths:
            if (output_dir / json_path.name).exists():
                clashes.append(json_path.name)
            if copy_images:
                # the image that would be copied alongside this JSON
                with open(json_path, "r") as f:
                    image_name = Path(json.load(f).get("imagePath", "")).name
                if image_name and (output_dir / image_name).exists():
                    clashes.append(image_name)
        if clashes:
            raise FileExistsError(
                f"{len(clashes)} file(s) already exist in {output_dir} "
                f"(nothing was written): {sorted(set(clashes))}"
            )

    # Counters for the summary printed at the end
    converted = Counter()    # (old_label, new_label) -> count
    unmapped = Counter()     # labels found that weren't in label_map
    skipped = []             # files left alone because they already existed
    n_files = 0

    # Loop over every JSON file in the input folder
    for json_path in json_paths:
        out_path = output_dir / json_path.name

        # In "skip" mode, leave existing output files untouched
        if on_exists == "skip" and out_path.exists():
            skipped.append(json_path.name)
            continue

        with open(json_path, "r") as f:
            data = json.load(f)

        # Swap the label on every shape (bounding box) in the file
        for shape in data.get("shapes", []):
            old_label = shape["label"]
            if old_label in label_map:
                new_label = label_map[old_label]
                shape["label"] = new_label
                converted[(old_label, new_label)] += 1
            else:
                # Label not in the map: keep it as-is but record it
                unmapped[old_label] += 1

        # Save under the same filename in the output folder
        with open(out_path, "w") as f:
            json.dump(data, f, indent=2)
        n_files += 1

        # Optionally copy the matching image across so LabelMe can open it
        if copy_images and data.get("imagePath"):
            img_path = input_dir / Path(data["imagePath"]).name
            img_out = output_dir / img_path.name
            if on_exists == "skip" and img_out.exists():
                skipped.append(img_path.name)
            elif img_path.exists():
                shutil.copy2(img_path, img_out)

    # ---- Summary ----
    print(f"Processed {n_files} JSON file(s)")
    print(f"  from: {input_dir}")
    print(f"  to:   {output_dir}\n")
    print("Label conversions:")
    for (old, new), count in sorted(converted.items()):
        print(f"  '{old}' -> '{new}': {count} label(s)")
    if skipped:
        print(f"\nSkipped {len(skipped)} file(s) that already existed in the output folder:")
        for name in skipped:
            print(f"  {name}")
    if unmapped:
        print("\nWARNING - labels found that were NOT in label_map (left unchanged):")
        for label, count in sorted(unmapped.items()):
            print(f"  '{label}': {count} label(s)")

## Case 1: single class — `WH` → `waterhole`

Files come from `training/output/WH` and are saved to `training/output/waterhole_river`.

In [ ]:
# Straight one-to-one switch: every 'WH' label becomes 'waterhole'
relabel_json_dir(
    input_dir="training/output/WH",
    output_dir="training/output/waterhole_river",
    label_map={
        "WH": "waterhole",
    },
    copy_images=True,  # copy the PNGs too so LabelMe can open the new JSONs
    on_exists="skip"
)

## Case 2: multiple classes — all waterhole types → `waterhole`

The JSONs in `training/output/WH_wet_dry_swamp_sink` use the labels `Dry_WH`, `WH_wet`, `WH_swamp` and `WH_sink` (checked against the actual files). These are all collapsed to a single `waterhole` class and saved to the same `waterhole_river` folder.

If you ever want a many-to-*two* mapping instead (e.g. keep sinkholes separate), just point some keys at a different value — for example `"WH_sink": "sinkhole"`.

In [ ]:
# Many-to-one mapping: collapse all waterhole sub-classes into 'waterhole'
relabel_json_dir(
    input_dir="training/output/WH_wet_dry_swamp_sink",
    output_dir="training/output/waterhole_river",
    label_map={
        "WH": "waterhole",        # in case any plain 'WH' labels are present
        "Dry_WH": "waterhole",
        "WH_wet": "waterhole",
        "WH_swamp": "waterhole",
        "WH_sink": "waterhole",
    },
    copy_images=True,
    on_exists="skip"
)